# Stage 11C-R — Manual Official Receipt Validation and Typed-Hold Readjudication

This development-only recovery notebook validates the five original manual official receipts acquired under Protocol Amendment A1. It inherits the sealed amended Stage 11C hold, fingerprints every received file, audits archive safety and content structure, derives label and patient/lesion grouping evidence without looking at model outcomes, and re-adjudicates each original dataset as `QUALIFY`, typed `HOLD`, or `EXCLUDE`.

The notebook cannot use embeddings, source AUC, transfer performance, DDO2, Stage 12, reserves, or locked-blind assets. Four independently qualified domains are required before Stage 11D-R can be authorised. A manual receipt is evidence of acquisition only; it is not automatically evidence of scientific eligibility.


In [1]:
# @title 11C-R-0. Mount Drive, verify sealed parent lineage, and freeze scope
import hashlib, json, os, re, shutil, zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    pass

DEFAULT_ROOT = Path('/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability') if IN_COLAB else Path('/tmp/Cross-Modal_Diagnostic_Observability')
PROJECT_ROOT = Path(os.environ.get('CDO_PROJECT_ROOT', str(DEFAULT_ROOT)))
CODE_ROOT = PROJECT_ROOT/'05_Code'/'Cross_Modal'
CM_ROOT = PROJECT_ROOT/'06_Data_Records'/'Cross_Modal'
PARENT_ROOT = CM_ROOT/'Stage11C_Amended_Original_First_Official_Route_Recovery_And_Outcome_Free_Reserve_Screening_v0.1'
ROOT = CM_ROOT/'Stage11C-R_Manual_Official_Receipt_Validation_And_Typed_Hold_Readjudication_v0.1'
RECEIPT_ROOT = PROJECT_ROOT/'00_Data_Acquisition'/'Stage11C_Manual_Official_Receipts'
P0,P1,P2,P3,P4,P5 = [ROOT/x for x in ['00_Protocol','01_Receipt_Integrity','02_Schema_And_Grouping','03_Readjudication','04_Firewall','05_Results']]
for p in [CODE_ROOT,P0,P1,P2,P3,P4,P5]: p.mkdir(parents=True,exist_ok=True)

NOTEBOOK_NAME='CrossModal_Stage11C-R_Manual_Official_Receipt_Validation_And_Typed_Hold_Readjudication_v0.1.ipynb'
NOTEBOOK_PATH=CODE_ROOT/NOTEBOOK_NAME
PARENT_FINAL=PARENT_ROOT/'05_Results'/'Stage11C_Amended_Recovery_And_Reserve_Screening_Complete_v0.1.json'
PARENT_OUTCOME=PARENT_ROOT/'01_Original_Official_Route_Recovery'/'Stage11C_Original_Recovery_Outcome_v0.1.csv'
PARENT_HANDOFF=PARENT_ROOT/'03_Final_Development_Roster'/'Stage11C_Stage11D_Prefit_Handoff_v0.1.json'
EXPECTED_PARENT_FINAL='05e0288a02af5255ce0335d6da74f78483c51ee8c9f2b63de60a6dd2a6fb1ada'
DATASETS=['BUS_BRA_2024','BUSI_WHU_2025_V3','BREAST_LESIONS_USG_2024','BUS_UCLM_2025_V3','RODRIGUES_BUI_2017']
FOLDERS={'BUS_BRA_2024':'BUS_BRA','BUSI_WHU_2025_V3':'BUSI_WHU','BREAST_LESIONS_USG_2024':'BREAST_LESIONS_USG','BUS_UCLM_2025_V3':'BUS_UCLM','RODRIGUES_BUI_2017':'RODRIGUES_BUI'}
LOCKED=['BUSI_CAIRO_2019','OASBUD_2017','DERM7PT_2019']
MINIMUM=4

PROTOCOL=P0/'Stage11C-R_Protocol_Seal_v0.1.json'
PARENT_COMMIT=P0/'Stage11C-R_Parent_Input_Commitment_v0.1.csv'
RECEIPT_LEDGER=P1/'Stage11C-R_Manual_Official_Receipt_Ledger_v0.1.csv'
ARCHIVE_AUDIT=P1/'Stage11C-R_Archive_Safety_And_Integrity_Audit_v0.1.csv'
CONTENT_AUDIT=P2/'Stage11C-R_Content_Label_And_Grouping_Audit_v0.1.csv'
DECISIONS=P3/'Stage11C-R_Typed_Hold_Readjudication_v0.1.csv'
ROSTER=P3/'Stage11C-R_Qualified_Development_Roster_v0.1.csv'
HANDOFF=P3/'Stage11C-R_Stage11D-R_Handoff_v0.1.json'
FIREWALL=P4/'Stage11C-R_Independent_Validity_And_Firewall_Checks_v0.1.csv'
REPORT=P5/'Stage11C-R_Manual_Receipt_Readjudication_Report_v0.1.md'
MANIFEST=P5/'Stage11C-R_Output_Integrity_Manifest_v0.1.csv'
FINAL=P5/'Stage11C-R_Complete_v0.1.json'
RUNTIME=P5/'Stage11C-R_Runtime_State_v0.1.json'

def now(): return datetime.now(timezone.utc).isoformat()
def sha_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
def sha_json(x): return hashlib.sha256(json.dumps(x,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
def canon(df): return df.to_csv(index=False,lineterminator='\n',float_format='%.12g')
def write_text(p,s):
    p=Path(p)
    if p.exists(): assert p.read_text(encoding='utf-8')==s, f'Replay mismatch: {p}'
    else: p.write_text(s,encoding='utf-8')
def write_csv(p,d): write_text(p,canon(d))
def write_json(p,x): write_text(p,json.dumps(x,indent=2,ensure_ascii=False)+'\n')
def verify_self(p,field,expected=None):
    x=json.loads(Path(p).read_text(encoding='utf-8')); claimed=x[field]; y=dict(x); y.pop(field)
    assert sha_json(y)==claimed, f'Self-hash mismatch: {p}'
    if expected: assert claimed==expected, f'Unexpected parent hash: {p}'
    return x

required=[NOTEBOOK_PATH,PARENT_FINAL,PARENT_OUTCOME,PARENT_HANDOFF]
missing=[str(p) for p in required if not p.is_file()]
assert not missing, 'Missing sealed inputs:\n'+'\n'.join(missing)
parent=verify_self(PARENT_FINAL,'final_record_sha256',EXPECTED_PARENT_FINAL)
parent_handoff=verify_self(PARENT_HANDOFF,'handoff_sha256')
assert parent['decision']=='HOLD_STAGE11C_INSUFFICIENT_PREFIT_QUALIFIED_DOMAINS_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED'
assert parent['stage11d_authorised'] is False and parent['locked_blind_assets_touched'] is False
parent_outcome=pd.read_csv(PARENT_OUTCOME)
assert parent_outcome.sort_values('original_priority')['dataset_id'].tolist()==DATASETS
REPLAY=FINAL.is_file()
commit=pd.DataFrame([{'role':p.name,'relative_path':str(p.relative_to(PROJECT_ROOT)),'size_bytes':p.stat().st_size,'sha256':sha_file(p)} for p in required[1:]])
write_csv(PARENT_COMMIT,commit)
spec={'scope':'FIVE_ORIGINAL_MANUAL_OFFICIAL_RECEIPTS_ONLY','datasets':DATASETS,'minimum_qualified_domains':MINIMUM,'receipt_rule':'original bytes, no rename required, no provider re-download','gates':['receipt integrity','safe archive','released diagnostic label semantics','patient or lesion grouping','provenance and independent-domain evidence'],'prohibited':['reserves','embedding','source AUC','transfer performance','DDO2','Stage12','locked-blind metadata/images/labels']}
payload={'stage':'Stage11C-R','version':'0.1','parent_stage11c_final_sha256':parent['final_record_sha256'],'parent_stage11c_outcome_sha256':sha_file(PARENT_OUTCOME),'parent_stage11c_handoff_sha256':parent_handoff['handoff_sha256'],'analysis_spec':spec}
if REPLAY:
    seal=verify_self(PROTOCOL,'seal_sha256')
    for k,v in payload.items(): assert seal[k]==v
else:
    seal=dict(payload); seal['sealed_utc']=now(); seal['seal_sha256']=sha_json(seal); write_json(PROTOCOL,seal)
runtime={'stage':'Stage11C-R','replay_mode':REPLAY,'manual_receipts_only':True,'provider_requests_made':False,'reserves_accessed':False,'embeddings_computed':False,'performance_evaluated':False,'ddo2_fitted':False,'stage12_authorised':False,'locked_blind_assets_touched':False}
print('Parent Stage11C verified:',parent['final_record_sha256']); print('Protocol seal / Replay:',seal['seal_sha256'],REPLAY)


Mounted at /content/drive
Parent Stage11C verified: 05e0288a02af5255ce0335d6da74f78483c51ee8c9f2b63de60a6dd2a6fb1ada
Protocol seal / Replay: 3dfff3bd0ba03a5d83f9cdfe177753c66d353d66454a79800e943fcd6e56df0f False


In [2]:
# @title 11C-R-1. Fingerprint received files and audit archive safety without extraction
IMAGE_EXT={'.png','.jpg','.jpeg','.bmp','.tif','.tiff','.dcm'}
TABLE_EXT={'.csv','.xlsx','.xls','.json','.txt','.xml'}
rows=[]; archive_rows=[]; members_by_dataset={}
for dataset_id in DATASETS:
    folder=RECEIPT_ROOT/FOLDERS[dataset_id]
    files=sorted([p for p in folder.iterdir() if p.is_file() and not p.name.startswith('.')]) if folder.is_dir() else []
    for p in files:
        rows.append({'dataset_id':dataset_id,'receipt_folder':FOLDERS[dataset_id],'file_name':p.name,'size_bytes':p.stat().st_size,'sha256':sha_file(p),'extension':p.suffix.lower(),'received':True})
    member_rows=[]
    for p in files:
        if p.suffix.lower()=='.zip':
            safe=True; bad=[]; encrypted=False; total=0; names=[]
            try:
                with zipfile.ZipFile(p) as z:
                    for info in z.infolist():
                        q=PurePosixPath(info.filename.replace('\\','/')); parts=q.parts
                        member_safe=not q.is_absolute() and '..' not in parts and not (parts and re.match(r'^[A-Za-z]:',parts[0]))
                        safe &= member_safe
                        if not member_safe: bad.append(info.filename)
                        encrypted |= bool(info.flag_bits & 1); total += int(info.file_size); names.append(info.filename)
                        member_rows.append({'archive_name':p.name,'member_name':info.filename,'member_size':int(info.file_size),'suffix':Path(info.filename).suffix.lower()})
                    archive_rows.append({'dataset_id':dataset_id,'archive_name':p.name,'archive_readable':True,'member_count':len(names),'uncompressed_bytes':total,'path_traversal_safe':safe,'encrypted':encrypted,'bad_member_count':len(bad),'error':''})
            except Exception as e:
                archive_rows.append({'dataset_id':dataset_id,'archive_name':p.name,'archive_readable':False,'member_count':0,'uncompressed_bytes':0,'path_traversal_safe':False,'encrypted':False,'bad_member_count':0,'error':type(e).__name__+': '+str(e)[:300]})
    members_by_dataset[dataset_id]=pd.DataFrame(member_rows)
ledger=pd.DataFrame(rows,columns=['dataset_id','receipt_folder','file_name','size_bytes','sha256','extension','received'])
archives=pd.DataFrame(archive_rows,columns=['dataset_id','archive_name','archive_readable','member_count','uncompressed_bytes','path_traversal_safe','encrypted','bad_member_count','error'])
if REPLAY:
    assert canon(pd.read_csv(RECEIPT_LEDGER))==canon(ledger); assert canon(pd.read_csv(ARCHIVE_AUDIT))==canon(archives)
else:
    write_csv(RECEIPT_LEDGER,ledger); write_csv(ARCHIVE_AUDIT,archives)
assert set(ledger.dataset_id)==set(DATASETS), 'At least one receipt folder is empty'
assert not ledger.file_name.str.lower().str.contains('|'.join(LOCKED),regex=True).any()
display(ledger[['dataset_id','file_name','size_bytes','sha256']])


,dataset_id,file_name,size_bytes,sha256
0,BUS_BRA_2024,BUSBRA.zip,133917740,ba3e6ed19cc37c682d8d39e25435bbf8a555a12cb7e641...
1,BUSI_WHU_2025_V3,BUSI_WHU Breast Cancer Ultrasound Image Datas...,127397821,9fcda99c284ba5ebcda5bb2761af68b32d9c8a7070bd57...
2,BREAST_LESIONS_USG_2024,BrEaST-Lesions-USG-clinical-data-Dec-15-2023.xlsx,40173,89a8874496a6f1f93960390b963f7369b90f7e72ee08a7...
3,BREAST_LESIONS_USG_2024,BrEaST-Lesions_USG-images_and_masks-Dec-15-202...,69862076,c32e49cdfa54042e065582d0cb35978b5fc5e5b5e461b3...
4,BUS_UCLM_2025_V3,BUS-UCLM Breast ultrasound lesion segmentation...,336180434,985cb6f399ba65087cbb6d3fa7ee775ad7ea30d69f1dbf...
5,RODRIGUES_BUI_2017,Breast Ultrasound Image.zip,3194579,fc727f96de14dfb7dd45d3b33f297c748932fd09323302...


In [3]:
# @title 11C-R-2. Audit released labels and patient/lesion grouping evidence
def member_names(d):
    m=members_by_dataset[d]
    return m.member_name.astype(str).tolist() if len(m) else []
def count_ext(names,exts): return sum(Path(x).suffix.lower() in exts for x in names)
def tokens(names,patterns):
    out=set()
    for n in names:
        for pat in patterns:
            m=re.search(pat,n,re.I)
            if m: out.add(m.group(1) if m.groups() else m.group(0))
    return out
def label_hits(names):
    s='\n'.join(names).lower()
    return sorted([x for x in ['benign','malignant','normal','birads','bi-rads','pathology','histology'] if x in s])

aud=[]
for d in DATASETS:
    names=member_names(d); lower=[x.lower() for x in names]
    images=count_ext(names,IMAGE_EXT); tables=count_ext(names,TABLE_EXT)
    labels=label_hits(names); group_ids=set(); grouping_basis=''
    if d=='BUS_BRA_2024':
        group_ids=tokens(names,[r'(?:patient|case|subject|pat)[_ -]?(\d+)',r'/([A-Z]?\d{3,})[_/ -]'])
        grouping_basis='released patient/case token in archive path or manifest'
    elif d=='BUSI_WHU_2025_V3':
        group_ids=tokens(names,[r'(?:patient|case|subject|pat)[_ -]?(\d+)',r'/([A-Z]?\d{3,})[_/ -]'])
        grouping_basis='released case token in archive hierarchy'
    elif d=='BREAST_LESIONS_USG_2024':
        # The separately released TCIA clinical workbook is mandatory grouping/label evidence.
        xlsx=next((RECEIPT_ROOT/FOLDERS[d]/x for x in ledger.loc[(ledger.dataset_id==d)&(ledger.extension=='.xlsx'),'file_name']),None)
        sheets=[]; cols=[]; nrows=0
        if xlsx and xlsx.is_file():
            book=pd.ExcelFile(xlsx); sheets=book.sheet_names
            for sh in sheets:
                t=pd.read_excel(xlsx,sheet_name=sh); nrows+=len(t); cols += [str(c) for c in t.columns]
                for c in t.columns:
                    if re.search(r'patient|subject|case|anon|id',str(c),re.I): group_ids.update(t[c].dropna().astype(str).tolist())
            labels += [c for c in cols if re.search(r'patholog|diagnos|benign|malignant|bi.?rads',c,re.I)]
        grouping_basis='TCIA clinical workbook patient/case identifier column'
        tables += int(bool(sheets))
    elif d=='BUS_UCLM_2025_V3':
        group_ids=tokens(names,[r'(?:patient|case|subject|pat)[_ -]?(\d+)',r'/([A-Z]?\d{3,})[_/ -]'])
        grouping_basis='released patient/case token in archive hierarchy'
    elif d=='RODRIGUES_BUI_2017':
        group_ids=tokens(names,[r'(?:patient|case|subject|pat)[_ -]?(\d+)',r'/([A-Z]?\d{3,})[_/ -]'])
        grouping_basis='released patient/case token if present; class folder alone is not grouping'
    diagnostic_label_evidence=bool(labels)
    grouping_proven=len(group_ids)>=2
    archive_ok=(len(archives[archives.dataset_id==d])>0 and archives.loc[archives.dataset_id==d,'archive_readable'].astype(bool).all() and archives.loc[archives.dataset_id==d,'path_traversal_safe'].astype(bool).all() and not archives.loc[archives.dataset_id==d,'encrypted'].astype(bool).any())
    aud.append({'dataset_id':d,'receipt_file_count':int((ledger.dataset_id==d).sum()),'archive_integrity_passed':bool(archive_ok),'image_like_member_count':images,'table_or_metadata_member_count':tables,'diagnostic_label_evidence':diagnostic_label_evidence,'label_evidence_summary':' | '.join(map(str,labels))[:500],'patient_or_lesion_grouping_proven':grouping_proven,'unique_group_ids_detected':len(group_ids),'grouping_basis':grouping_basis,'model_outcome_used':False})
content=pd.DataFrame(aud)
if REPLAY: assert canon(pd.read_csv(CONTENT_AUDIT))==canon(content)
else: write_csv(CONTENT_AUDIT,content)
display(content)


,dataset_id,receipt_file_count,archive_integrity_passed,image_like_member_count,table_or_metadata_member_count,diagnostic_label_evidence,label_evidence_summary,patient_or_lesion_grouping_proven,unique_group_ids_detected,grouping_basis,model_outcome_used
0,BUS_BRA_2024,1,True,3750,4,False,,False,0,released patient/case token in archive path or...,False
1,BUSI_WHU_2025_V3,1,True,0,1,False,,False,0,released case token in archive hierarchy,False
2,BREAST_LESIONS_USG_2024,2,True,522,1,True,BIRADS | Diagnosis,True,256,TCIA clinical workbook patient/case identifier...,False
3,BUS_UCLM_2025_V3,1,True,1366,0,False,,False,0,released patient/case token in archive hierarchy,False
4,RODRIGUES_BUI_2017,1,True,0,0,False,,False,0,released patient/case token if present; class ...,False


In [4]:
# @title 11C-R-3. Apply conservative typed adjudication and freeze the recovered roster
rows=[]
for i,d in enumerate(DATASETS,1):
    a=content.set_index('dataset_id').loc[d]
    receipt_ok=int(a.receipt_file_count)>0
    if not receipt_ok: decision,status,reason='HOLD','HOLD_RECEIPT_MISSING','No file in frozen manual receipt folder'
    elif not bool(a.archive_integrity_passed): decision,status,reason='HOLD','HOLD_ARCHIVE_INTEGRITY','Archive unreadable, unsafe, or encrypted'
    elif int(a.image_like_member_count)==0: decision,status,reason='HOLD','HOLD_PIXEL_ASSET_STRUCTURE','No image-like member was identified'
    elif not bool(a.diagnostic_label_evidence): decision,status,reason='HOLD','HOLD_LABEL_SEMANTICS','Released receipt does not expose auditable diagnostic label semantics'
    elif not bool(a.patient_or_lesion_grouping_proven): decision,status,reason='HOLD','HOLD_GROUPING_EVIDENCE','No deterministic patient/lesion grouping key with at least two groups was established'
    else: decision,status,reason='QUALIFY','QUALIFY_PREFIT_MANUAL_OFFICIAL_RECEIPT','Receipt, archive, labels, and grouping passed; cross-roster pixel dedup remains mandatory before fitting'
    rows.append({'original_priority':i,'dataset_id':d,'decision':decision,'status':status,'reason':reason,'official_manual_receipt_verified':receipt_ok,'archive_integrity_passed':bool(a.archive_integrity_passed),'diagnostic_label_evidence':bool(a.diagnostic_label_evidence),'patient_or_lesion_grouping_proven':bool(a.patient_or_lesion_grouping_proven),'performance_evaluated':False})
decisions=pd.DataFrame(rows)
qualified=decisions[decisions.decision=='QUALIFY'].copy()
roster=qualified[['original_priority','dataset_id']].copy(); roster.insert(0,'roster_position',range(1,len(roster)+1)); roster['role']='RECOVERED_ORIGINAL_DEVELOPMENT_EXTENSION'; roster['source_axis_fitted']=False; roster['stage12_role_permitted']=False
authorised=len(roster)>=MINIMUM
decision='SEAL_STAGE11C_R_AUTHORISE_STAGE11D_R_DEDUP_AND_GROUPED_SPLIT_FREEZE_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED' if authorised else 'HOLD_STAGE11C_R_INSUFFICIENT_QUALIFIED_MANUAL_RECEIPTS_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED'
if REPLAY:
    assert canon(pd.read_csv(DECISIONS))==canon(decisions); assert canon(pd.read_csv(ROSTER))==canon(roster)
    handoff=verify_self(HANDOFF,'handoff_sha256')
else:
    write_csv(DECISIONS,decisions); write_csv(ROSTER,roster)
    handoff={'stage':'Stage11C-R','target':'Stage11D-R','decision':decision,'protocol_seal_sha256':seal['seal_sha256'],'parent_stage11c_final_sha256':parent['final_record_sha256'],'receipt_ledger_sha256':sha_file(RECEIPT_LEDGER),'content_audit_sha256':sha_file(CONTENT_AUDIT),'readjudication_sha256':sha_file(DECISIONS),'qualified_roster_sha256':sha_file(ROSTER),'qualified_dataset_ids':roster.dataset_id.tolist(),'qualified_domain_count':len(roster),'minimum_feasible_domains':MINIMUM,'stage11d_r_authorised':authorised,'next_stage_boundary':'cross-roster pixel deduplication, image-level patient-group joins, grouped split freeze, then development-only source recoverability','stage12_authorised':False,'ddo2_fitted':False,'locked_blind_assets_touched':False}
    handoff['handoff_sha256']=sha_json(handoff); write_json(HANDOFF,handoff)
display(decisions[['dataset_id','decision','status','reason']]); print('Qualified roster:',roster.dataset_id.tolist()); print('Stage11D-R authorised:',authorised)


,dataset_id,decision,status,reason
0,BUS_BRA_2024,HOLD,HOLD_LABEL_SEMANTICS,Released receipt does not expose auditable dia...
1,BUSI_WHU_2025_V3,HOLD,HOLD_PIXEL_ASSET_STRUCTURE,No image-like member was identified
2,BREAST_LESIONS_USG_2024,QUALIFY,QUALIFY_PREFIT_MANUAL_OFFICIAL_RECEIPT,"Receipt, archive, labels, and grouping passed;..."
3,BUS_UCLM_2025_V3,HOLD,HOLD_LABEL_SEMANTICS,Released receipt does not expose auditable dia...
4,RODRIGUES_BUI_2017,HOLD,HOLD_PIXEL_ASSET_STRUCTURE,No image-like member was identified


Qualified roster: ['BREAST_LESIONS_USG_2024']
Stage11D-R authorised: False


In [5]:
# @title 11C-R-4. Run independent replay, leakage, and authority checks
checks=[]
def ck(name,passed,evidence): checks.append({'check':name,'passed':bool(passed),'evidence':str(evidence)[:1000]})
ck('sealed parent exact',parent['final_record_sha256']==EXPECTED_PARENT_FINAL,parent['final_record_sha256'])
ck('parent remains hold',parent['stage11d_authorised'] is False,parent['decision'])
ck('five originals exact order',decisions.dataset_id.tolist()==DATASETS,decisions.dataset_id.tolist())
ck('all receipt folders inside frozen root',all((RECEIPT_ROOT/FOLDERS[d]).is_dir() for d in DATASETS),RECEIPT_ROOT)
ck('no locked-blind token in receipt ledger',not ledger.file_name.str.lower().str.contains('|'.join(LOCKED),regex=True).any(),ledger.file_name.tolist())
ck('no reserve accessed',runtime['reserves_accessed'] is False,'false')
ck('no provider request',runtime['provider_requests_made'] is False,'false')
ck('no performance input',not decisions.performance_evaluated.astype(bool).any() and not content.model_outcome_used.astype(bool).any(),'all false')
ck('qualify requires all prefit gates',all(bool(r.archive_integrity_passed and r.diagnostic_label_evidence and r.patient_or_lesion_grouping_proven) for r in decisions[decisions.decision=='QUALIFY'].itertuples()),qualified.dataset_id.tolist())
ck('roster equals qualified originals',roster.dataset_id.tolist()==qualified.dataset_id.tolist(),roster.dataset_id.tolist())
ck('minimum rule exact',bool(handoff['stage11d_r_authorised'])==(len(roster)>=MINIMUM),len(roster))
ck('no embedding or fit',runtime['embeddings_computed'] is False and runtime['performance_evaluated'] is False,'false')
ck('DDO2 and Stage12 prohibited',runtime['ddo2_fitted'] is False and handoff['stage12_authorised'] is False,'false')
ck('locked blind untouched',runtime['locked_blind_assets_touched'] is False and handoff['locked_blind_assets_touched'] is False,'false')
ck('handoff self hash',sha_json({k:v for k,v in handoff.items() if k!='handoff_sha256'})==handoff['handoff_sha256'],handoff['handoff_sha256'])
validity=pd.DataFrame(checks)
if REPLAY: assert canon(pd.read_csv(FIREWALL))==canon(validity)
else: write_csv(FIREWALL,validity)
failed=validity.loc[~validity.passed,'check'].tolist(); assert not failed,'Validity failure: '+'; '.join(failed)
print(f'Independent checks: {validity.passed.sum()}/{len(validity)} passed')


Independent checks: 15/15 passed


In [6]:
# @title 11C-R-5. Seal report, integrity manifest, final record, and print handoff
q=int((decisions.decision=='QUALIFY').sum()); h=int((decisions.decision=='HOLD').sum()); e=int((decisions.decision=='EXCLUDE').sum())
next_step='BUILD_STAGE11D_R_CROSS_ROSTER_DEDUP_GROUPED_SPLIT_FREEZE_AND_SOURCE_RECOVERABILITY' if authorised else 'REVIEW_REMAINING_TYPED_HOLDS_AND_PROVIDER_EVIDENCE_WITHOUT_PERFORMANCE_BASED_SUBSTITUTION'
report=f"""# Stage 11C-R report

## Answer first

- Manual official receipt folders present: **{content.receipt_file_count.gt(0).sum()}/5**.
- Qualified / held / excluded: **{q} / {h} / {e}**.
- Minimum feasible domains: **{MINIMUM}**.
- Stage 11D-R authorised: **{authorised}**.
- Decision: `{decision}`.

## Dataset adjudication

{decisions.fillna('').to_markdown(index=False)}

## Method boundary

No reserve, embedding, source AUC, transfer result, DDO2 operation, Stage 12 operation, or locked-blind asset was used. A later Stage 11D-R must still perform cross-roster pixel deduplication and freeze patient-grouped splits before fitting.
"""
if REPLAY: assert REPORT.read_text(encoding='utf-8')==report
else: write_text(REPORT,report)
tracked=[PROTOCOL,PARENT_COMMIT,RECEIPT_LEDGER,ARCHIVE_AUDIT,CONTENT_AUDIT,DECISIONS,ROSTER,HANDOFF,FIREWALL,REPORT]
manifest=pd.DataFrame([{'relative_path':str(p.relative_to(ROOT)),'size_bytes':p.stat().st_size,'sha256':sha_file(p)} for p in tracked])
if REPLAY:
    old=pd.read_csv(MANIFEST); assert canon(old)==canon(manifest)
    for r in old.itertuples():
        p=ROOT/r.relative_path; assert p.is_file() and p.stat().st_size==int(r.size_bytes) and sha_file(p)==r.sha256
else: write_csv(MANIFEST,manifest)
payload={'stage':'Stage11C-R','version':'0.1','decision':decision,'protocol_seal_sha256':seal['seal_sha256'],'parent_stage11c_final_sha256':parent['final_record_sha256'],'manual_receipt_ledger_sha256':sha_file(RECEIPT_LEDGER),'content_audit_sha256':sha_file(CONTENT_AUDIT),'typed_readjudication_sha256':sha_file(DECISIONS),'qualified_roster_sha256':sha_file(ROSTER),'stage11d_r_handoff_sha256':handoff['handoff_sha256'],'output_integrity_manifest_sha256':sha_file(MANIFEST),'manual_receipt_domains_present':int(content.receipt_file_count.gt(0).sum()),'qualified_domains':q,'held_domains':h,'excluded_domains':e,'minimum_feasible_domains':MINIMUM,'stage11d_r_authorised':authorised,'stage12_authorised':False,'ddo2_fitted':False,'locked_blind_assets_touched':False,'next_step':next_step}
if REPLAY:
    final=verify_self(FINAL,'final_record_sha256')
    for k,v in payload.items(): assert final[k]==v,f'Replay final changed: {k}'
else:
    final=dict(payload); final['completed_utc']=now(); final['final_record_sha256']=sha_json(final); write_json(FINAL,final)
runtime.update({'completed':True,'decision':decision,'final_record_sha256':final['final_record_sha256'],'last_updated_utc':now()}); RUNTIME.write_text(json.dumps(runtime,indent=2)+'\n')
print('================ STAGE 11C-R COMPLETE ================')
print('Manual official receipt domains present:',f"{int(content.receipt_file_count.gt(0).sum())}/5")
print('Qualified / held / excluded:',f'{q}/{h}/{e}')
print('Qualified development roster / minimum feasible:',f'{len(roster)}/{MINIMUM}')
print('Stage11D-R authorised:',authorised)
print('Decision:',decision)
print('Protocol seal:',seal['seal_sha256'])
print('Manual receipt ledger hash:',final['manual_receipt_ledger_sha256'])
print('Content audit hash:',final['content_audit_sha256'])
print('Typed readjudication hash:',final['typed_readjudication_sha256'])
print('Qualified roster hash:',final['qualified_roster_sha256'])
print('Stage11D-R handoff hash:',final['stage11d_r_handoff_sha256'])
print('Final record hash:',final['final_record_sha256'])
print('Next step:',next_step)


================ STAGE 11C-R COMPLETE ================
Manual official receipt domains present: 5/5
Qualified / held / excluded: 1/4/0
Qualified development roster / minimum feasible: 1/4
Stage11D-R authorised: False
Decision: HOLD_STAGE11C_R_INSUFFICIENT_QUALIFIED_MANUAL_RECEIPTS_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED
Protocol seal: 3dfff3bd0ba03a5d83f9cdfe177753c66d353d66454a79800e943fcd6e56df0f
Manual receipt ledger hash: 9711ff452cd343d0b2ba59b217f3c93cb9f81ed2d8f8dae42a424137de245a4f
Content audit hash: 1b6ae661cd19d76c2237e27d9d2ef0024c2d8424609b9b19c986199853d8c558
Typed readjudication hash: 6666d68c38159c4752812904a501b98375771ea488146c225999198500b706d2
Qualified roster hash: e31afdf638fe348f5eafc7e09f855811ff60f39f72928dff5410d9ef5ef559ba
Stage11D-R handoff hash: c1bb623753397764575030dccd779cb0dce55b19620519b006cca7cb69cf1c1f
Final record hash: d50440479a53b63cc22c8bd643089ce43545587f29b0062897f76b9e774ac768
Next step: REVIEW_REMAINING_TYPED_HOLDS_AND_PROVIDER_EVIDENCE_WITHO